In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


### Read in the (test/submission) data

In [ ]:
df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test.csv'))
display(df_test.head())

### Stage 1 Predictions -- Classifying Demand_Response_Flag

In [ ]:
def preprocess_data(df_in):
    df = df_in.copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['DOW'] = df['Timestamp'].dt.dayofweek
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # sin and cos transformation for cyclical features
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['DOW_sin'] = np.sin(2 * np.pi * df['DOW'] / 7)
    df['DOW_cos'] = np.cos(2 * np.pi * df['DOW'] / 7)

    # weather features
    def add_weather_features(df):
        t = df["Dry_Bulb_Temperature_C"]
        rad = df["Global_Horizontal_Radiation_W/m2"].clip(lower=0)
        df["HDD18"] = (18 - t).clip(lower=0)
        df["CDD22"] = (t - 22).clip(lower=0)
        df["TempC2"] = t**2
        df["rad_sqrt"] = np.sqrt(rad)
        df["rad_log1p"] = np.log1p(rad)
        df["is_daylight"] = (rad > 20).astype(int)
        # simple interactions
        df["TempC_x_daylight"] = t * df["is_daylight"]
        df["CDD22_x_daylight"] = df["CDD22"] * df["is_daylight"]
        df["TempC_x_hour_sin"] = t * df["Hour_sin"]
        df["TempC_x_hour_cos"] = t * df["Hour_cos"]
        return df
    df = add_weather_features(df)

    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([5, 6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2, 3]).astype(int)
    # Create hour of day categories
    df['Is_Afternoon'] = df['Hour'].isin(range(12, 18)).astype(int)
    df['Is_Evening'] = df['Hour'].isin(range(18, 24)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site'], inplace=True)
    return df

df = preprocess_data(df_test)

In [ ]:
# Load the trained model from file
from net_architecture import Net    # Define the neural network architecture: Just need to load architecture defined in net_architecture.py
input_dim = df.shape[1]             # Number of features
num_classes = 3                     # Number of classes in target variable
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('./models/phase1_nn_model.pth'))

# load the scaler, which should be ColumnTransformer type
# scaler is a combination of PowerTransformer (Yeo-Johnson transformation) & StandardScaler
scaler = joblib.load('./models/phase1_scaler.pkl')
print(scaler)

In [ ]:
# Convert DataFrame to torch tensor
X = scaler.transform(df)
X = torch.tensor(X, dtype=torch.float32)

# Set model to evaluation mode
model_loaded.eval()
with torch.no_grad():
    outputs = model_loaded(X)
    predictions = torch.argmax(outputs, dim=1)

# convert 2 in predictions to -1
predictions = np.where(predictions == 2, -1, predictions)

# Calculate the distribution of predictions classes
unique, counts = np.unique(predictions, return_counts=True)
distribution_pred = dict(zip(unique, counts))
print(distribution_pred)

In [ ]:
# Prepare data for Stage 2 predictions
df_stage2 = df_test.copy(deep=True)
df_stage2['Demand_Response_Flag'] = predictions

### Stage 2 Predictions - Demand_Response_Capacity_kW

In [ ]:
def preprocess_data(df_in):
    df = df_in.copy()
    # First, need to remove rows with Demand_Response_Flag = 0
    # df = df[df['Demand_Response_Flag'] != 0].copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['DOW'] = df['Timestamp'].dt.dayofweek
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # sin and cos transformation for cyclical features
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['DOW_sin'] = np.sin(2 * np.pi * df['DOW'] / 7)
    df['DOW_cos'] = np.cos(2 * np.pi * df['DOW'] / 7)

    # weather features
    def add_weather_features(df):
        t = df["Dry_Bulb_Temperature_C"]
        rad = df["Global_Horizontal_Radiation_W/m2"].clip(lower=0)
        df["HDD18"] = (18 - t).clip(lower=0)
        df["CDD22"] = (t - 22).clip(lower=0)
        df["TempC2"] = t**2
        df["rad_sqrt"] = np.sqrt(rad)
        df["rad_log1p"] = np.log1p(rad)
        df["is_daylight"] = (rad > 20).astype(int)
        # simple interactions
        df["TempC_x_daylight"] = t * df["is_daylight"]
        df["CDD22_x_daylight"] = df["CDD22"] * df["is_daylight"]
        df["TempC_x_hour_sin"] = t * df["Hour_sin"]
        df["TempC_x_hour_cos"] = t * df["Hour_cos"]
        return df
    df = add_weather_features(df)

    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([5, 6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2, 3]).astype(int)
    # Create hour of day categories
    df['Is_Afternoon'] = df['Hour'].isin(range(12, 18)).astype(int)
    df['Is_Evening'] = df['Hour'].isin(range(18, 24)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site'], inplace=True)
    return df

df = preprocess_data(df_stage2)
df.head()

In [ ]:
# Load models
clf_nz = joblib.load('./models/phase2_xgb_clf_nz_model.pkl')
reg_nz = joblib.load('./models/phase2_xgb_reg_nz_model.pkl')

# Prepare test features (X_test)
X_test = df.values

# Predict on real test set
p_nz = clf_nz.predict_proba(X_test)[:, 1]
mu_nz = reg_nz.predict(X_test)
y_pred_xgb = p_nz * mu_nz


In [ ]:
df_submission = df_test[['Site', 'Timestamp_Local']].copy(deep=True)
df_submission['Demand_Response_Flag'] = predictions
df_submission['Demand_Response_Capacity_kW'] = y_pred_xgb

# manually enforce 0 for Demand_Response_Capacity_kW if Demand_Response_Flag = 0
df_submission.loc[df_submission['Demand_Response_Flag'] == 0, 'Demand_Response_Capacity_kW'] = 0

df_submission.to_csv('./submissions/submission.csv', index=False)
print("Submission file created: submission.csv in ./submissions folder")

In [ ]:
display(df_submission.head())
print('submission file shape:', df_submission.shape)